# 16 · Trading agent on the initiation split, with a stay-flat fallback (E16)

Step 5 of the initiation plan. The agent reads E15's selected factors (re-tuned GRU, 3 seeds averaged,
`thesis_rl/initiation/e15/selected_mean3`) and is **re-tuned on the new validation period**. E6's settings were
tuned on the old split, whose validation overlaps the new test.

* **Agent:** Double DQN, the best algorithm in E6. Actions sell / hold / buy one step, positions −1 / 0 / +1,
  a decision every 5 minutes, 0.10 % fee + 0.02 % slippage per side, orders filled at the next bar.
* **State A only: market features, no LLM signals.** The agent sees its position, time in position and the 8
  compressed market factors (state size 10). Sentiment and risk are left out for now and can be added back later
  as states B / C / D with the same settings.
* **Tuning:** Optuna, 15 trials × 500k steps, state A, scored by validation Sharpe.
* **Final runs:** state A, 5 seeds, 2M steps, the tuned settings.
* **Checkpoint choice (validation):** the best validation Sharpe among checkpoints that trade at least 5 times.
* **Stay-flat fallback (new):** if no checkpoint qualifies, or the best one does not beat **both cash and buy and
  hold** on validation (its Sharpe must exceed 0 and buy and hold's validation Sharpe), the run stays in cash.
  Before this, it traded with the least-bad checkpoint even when that lost money. Each run's checkpoint numbers are kept too, so the table shows the agent with and without the fallback.
* **Baselines on validation:** buy and hold, always short, cash.

**Only validation is used here.** The test and stress_bear arrays are not even unzipped. Models are saved, and
the single final test (step 6) backtests them from the saved files (`runner.backtest_saved`).

Expected time on a T4: tuning about 3-4 h, final runs about 70 min each × 5, so about 9-10 h in total. Every
stage resumes after a disconnect: the Optuna study and finished runs are kept on Drive, so just run all again.

In [ ]:
# ---- setup: Drive, code, packages (identical in every notebook) -------------------------
BRANCH = "exp/14-initiation"
REPO = "https://github.com/taimoor-ahmed-1/FinAI-CryptoRaiders.git"
DRIVE_ROOT = "/content/drive/MyDrive/thesis_rl"      # data in, every output out - survives disconnects

import os, sys, subprocess
if "google.colab" in sys.modules:
    from google.colab import drive
    drive.mount("/content/drive")
    REPO_DIR = "/content/FinAI-CryptoRaiders"
    if not os.path.exists(REPO_DIR):
        # sparse clone: only experiments/ is checked out - the repo's data folders are never downloaded
        subprocess.run(["git", "clone", "--filter=blob:none", "--no-checkout", "--branch", BRANCH, REPO, REPO_DIR], check=True)
        subprocess.run(["git", "-C", REPO_DIR, "sparse-checkout", "set", "--no-cone", "/experiments/"], check=True)
    subprocess.run(["git", "-C", REPO_DIR, "fetch", "origin", BRANCH], check=True)
    subprocess.run(["git", "-C", REPO_DIR, "checkout", "-B", BRANCH, "FETCH_HEAD"], check=True)
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", "-r", f"{REPO_DIR}/experiments/requirements-colab.txt"], check=True)
    EXP = f"{REPO_DIR}/experiments"
else:                                                # local run from a checkout: notebooks/ -> experiments/
    EXP = os.path.abspath("..")
    DRIVE_ROOT = os.environ.get("THESIS_RL_ROOT", os.path.abspath("../../thesis_rl_outputs"))
sys.path.insert(0, EXP)
os.chdir(EXP)

import torch
from rltrade import paths, seeding
L = paths.layout(DRIVE_ROOT)
print("code:", BRANCH, seeding.git_commit(EXP), "| device:", "cuda - " + torch.cuda.get_device_name(0) if torch.cuda.is_available() else "cpu")
if not torch.cuda.is_available(): print('WARNING: no GPU - Runtime > Change runtime type > T4 GPU')

In [ ]:
seeding.record_environment(L.report / "environment", "16_initiation_agent")

In [ ]:
import json, zipfile, numpy as np, pandas as pd, matplotlib.pyplot as plt, optuna
from pathlib import Path
from rltrade.env import EnvConfig
from rltrade.runner import RunSpec, run, INVALID_SCORE
from rltrade import tune_rl, results as R
optuna.logging.set_verbosity(optuna.logging.WARNING)

INIT = L.root / "initiation"
OUT, REP = INIT / "agents", INIT / "report"
for p in (OUT, REP): p.mkdir(parents=True, exist_ok=True)
LOCAL = Path("/content/initiation_prepared") if "google.colab" in sys.modules else INIT / "prepared"
ZIP = INIT / "initiation_prepared.zip"
PREP = LOCAL / "fixed_plus"                         # only the market arrays (prices, signals) are read
NEEDED = [f"fixed_plus/market_{s}.npz" for s in ("train", "val")] + ["fixed_plus/meta.json"]
if not all((LOCAL / n).exists() for n in NEEDED):
    assert ZIP.exists(), f"upload initiation_prepared.zip to {ZIP}"
    with zipfile.ZipFile(ZIP) as z: z.extractall(LOCAL, members=NEEDED)   # test / stress_bear stay zipped
    print("unzipped", NEEDED, "->", LOCAL)

sel = json.loads((INIT / "selected_factors_e15.json").read_text())
F = Path(sel["factor_dir"])
if not F.exists():                                  # written on another machine: same place under initiation/
    parts = F.parts
    F = INIT.joinpath(*parts[parts.index("initiation") + 1:])
for s in ("train", "val"):
    f, m = np.load(F / f"factors_{s}.npy", mmap_mode="r"), np.load(PREP / f"market_{s}.npz")
    assert f.shape == (len(m["midpoint"]), 8), (s, f.shape)
    print(f"{s:5s} {f.shape[0]:7,d} bars | factors from {F.name} ({sel['choice']}, val MSE {sel['val_mse']:.5f})")

AGENT, EXP_NAME = "ddqn", "16_agents"
TUNE_STATE, STATES, SEEDS = "A", ("A",), [0, 1, 2, 3, 4]   # A = 8 market factors only, no sentiment / risk
N_TRIALS, TUNE_STEPS = 15, 500_000
TOTAL_STEPS, EVAL_EVERY = 2_000_000, 100_000
MIN_TRADES = 5                       # a checkpoint must trade at least this often on validation
FALLBACK = "cash"                    # ... and beat cash and buy and hold on validation, else the run stays flat
EVAL = ("val",)                      # the only split scored in this notebook
AUTO_DISCONNECT = True               # release the Colab runtime when the last cell has written its outputs

## 1 · Baselines on validation (seconds)

In [ ]:
for b in ("buy_and_hold", "always_short", "cash"):
    run(RunSpec(EXP_NAME, b, env=EnvConfig(state="A")), PREP, F, OUT, eval_splits=EVAL)

## 2 · Re-tune Double DQN on the new validation period (state A)

In [ ]:
st = tune_rl.tune_agent(AGENT, PREP, F, OUT, env_cfg=EnvConfig(state=TUNE_STATE), n_trials=N_TRIALS,
                        total_steps=TUNE_STEPS, eval_every=EVAL_EVERY, min_val_trades=MIN_TRADES, eval_splits=EVAL)
tr = st.trials_dataframe()
done = tr[tr.state == "COMPLETE"].copy()
show = ["number", "value", "user_attrs_val_sharpe", "user_attrs_val_cum_return_pct", "user_attrs_val_num_trades"] + \
       [c for c in done.columns if c.startswith("params_")]
print(f"{len(done)} complete, {(tr.state == 'PRUNED').sum()} pruned, "
      f"{(done.value > INVALID_SCORE).sum()} with an eligible checkpoint (>= {MIN_TRADES} validation trades)")
print(done.sort_values("value", ascending=False)[show].round(4).to_string(index=False))
if st.best_value > INVALID_SCORE:
    best_params = st.best_params
else:                                # no trial traded enough: take the best validation Sharpe and say so
    best_params = st.trials[int(done.sort_values("user_attrs_val_sharpe").number.iloc[-1])].params
    print("WARNING: no eligible trial - using the best validation Sharpe; expect stay-flat fallbacks")
AGENT_HP, ENV_OVER = tune_rl.params_to_hp(AGENT, best_params)
(OUT / "tuning" / "best_hp.json").write_text(json.dumps({"agent": AGENT, "params": best_params, "agent_hp": AGENT_HP,
    "env": ENV_OVER, "best_val_sharpe": st.best_value}, indent=2, default=list))
print("selected:", AGENT_HP, ENV_OVER)

## 3 · Final runs: state A, 5 seeds (about 70 min each)

In [ ]:
hp = json.loads((OUT / "tuning" / "best_hp.json").read_text())
AGENT_HP, ENV_OVER = hp["agent_hp"], hp["env"]
AGENT_HP["net_dims"] = tuple(AGENT_HP["net_dims"])
# the bar a run's checkpoint must clear on validation: cash (Sharpe 0) and buy and hold, whichever is higher
bh_id = RunSpec(EXP_NAME, "buy_and_hold", env=EnvConfig(state="A")).run_id
BH_VAL_SHARPE = json.loads((OUT / "models" / bh_id / "metrics.json").read_text())["val"]["sharpe"]
MIN_SCORE = max(0.0, BH_VAL_SHARPE)
print(f"a run must reach validation Sharpe > {MIN_SCORE:.3f} (cash 0, buy and hold {BH_VAL_SHARPE:+.3f}), else it stays flat")
for seed in SEEDS:
    for state in STATES:
        run(RunSpec(EXP_NAME, AGENT, seed=seed, env=EnvConfig(state=state, **ENV_OVER), agent_hp=AGENT_HP,
                    total_steps=TOTAL_STEPS, eval_every=EVAL_EVERY, min_val_trades=MIN_TRADES, fallback=FALLBACK,
                    fallback_min_score=MIN_SCORE),
            PREP, F, OUT, eval_splits=EVAL)

## 4 · Results on validation (test and stress_bear stay hidden)

In [ ]:
df = R.load_results(OUT)
df = df[df.experiment == EXP_NAME].copy()
ag, bl = df[df.agent == AGENT].sort_values(["state", "seed"]), df[df.agent != AGENT]
per_run = ag[["state", "seed", "best_step", "selection_eligible", "ckpt_val_sharpe", "ckpt_val_cum_return_pct",
              "ckpt_val_num_trades", "fallback_used", "val_sharpe", "val_cum_return_pct", "val_max_drawdown_pct",
              "val_num_trades", "minutes"]]
print("every run (ckpt_* = the selected checkpoint; val_* = what is traded after the fallback rule)")
print(per_run.round(3).to_string(index=False))

def ms(x, f="+.2f"): x = pd.to_numeric(x, errors="coerce"); return f"{x.mean():{f}} ± {x.std(ddof=1):.{f[-2]}f}" if len(x) > 1 else f"{x.mean():{f}}"
rows = []
for s, g in ag.groupby("state"):
    fb = g.fallback_used.astype(str).eq("True")
    rows.append({"policy": f"DDQN state {s}, checkpoint", "seeds": len(g), "val Sharpe": ms(g.ckpt_val_sharpe),
                 "val return %": ms(g.ckpt_val_cum_return_pct), "max DD %": "", "trades": ms(g.ckpt_val_num_trades, ".0f"), "stayed flat": ""})
    rows.append({"policy": f"DDQN state {s}, with fallback", "seeds": len(g), "val Sharpe": ms(g.val_sharpe),
                 "val return %": ms(g.val_cum_return_pct), "max DD %": ms(g.val_max_drawdown_pct, ".2f"),
                 "trades": ms(g.val_num_trades, ".0f"), "stayed flat": f"{int(fb.sum())} of {len(g)}"})
for _, r in bl.iterrows():
    rows.append({"policy": r.agent, "seeds": 1, "val Sharpe": f"{r.val_sharpe:+.2f}", "val return %": f"{r.val_cum_return_pct:+.2f}",
                 "max DD %": f"{r.val_max_drawdown_pct:.2f}", "trades": int(r.val_num_trades), "stayed flat": ""})
summary = pd.DataFrame(rows)
print("\nvalidation, mean ± std over seeds\n" + summary.to_string(index=False))
per_run.to_csv(REP / "e16_agents_val_runs.csv", index=False)
(REP / "e16_agents_val.md").write_text("# E16 trading agent, validation only\n\n" + summary.to_markdown(index=False)
    + "\n\n## Every run\n\n" + per_run.round(3).to_markdown(index=False) + "\n", encoding="utf-8")

In [ ]:
def curve(rid):
    f = OUT / "models" / rid / "val_backtest.npz"
    if not f.exists(): return None
    z = np.load(f); return R.as_datetime(z["time_ns"]), z["equity"][1:]
fig, axes = plt.subplots(1, len(STATES), figsize=(6.5 * len(STATES), 4), sharey=True)
for ax, s in zip(np.atleast_1d(axes), STATES):
    for _, r in ag[ag.state == s].iterrows():
        c = curve(r.run_id)
        if c: ax.plot(*c, lw=1, alpha=0.8, ls=":" if str(r.fallback_used) == "True" else "-", label=f"seed {r.seed}")
    for _, r in bl.iterrows():
        c = curve(r.run_id)
        if c: ax.plot(*c, lw=1.2, color={"buy_and_hold": "black", "always_short": "grey", "cash": "lightgrey"}[r.agent],
                      ls="--", label=r.agent)
    ax.set_title(f"DDQN state {s}, validation (dotted = stayed flat)"); ax.grid(alpha=0.3); ax.tick_params(axis="x", rotation=30)
np.atleast_1d(axes)[0].set_ylabel("equity (start = 1)"); np.atleast_1d(axes)[-1].legend(fontsize=7, loc="upper left")
fig.tight_layout(); fig.savefig(REP / "e16_val_equity.png", dpi=130); plt.show()

In [ ]:
(INIT / "selected_agent_e16.json").write_text(json.dumps({
    "agent": AGENT, "agent_hp": AGENT_HP, "env": ENV_OVER, "states": list(STATES), "seeds": SEEDS,
    "factor_dir": str(F), "prepared": "initiation_prepared.zip: fixed_plus (market arrays)",
    "selection": f"best validation Sharpe among checkpoints with >= {MIN_TRADES} validation trades",
    "fallback": f"{FALLBACK} when no checkpoint qualifies or its validation Sharpe <= {MIN_SCORE:.4f} "
                f"(max of cash 0 and buy and hold {BH_VAL_SHARPE:+.4f})",
    "runs": sorted(ag.run_id), "models": str(OUT / "models"),
    "next": "step 6: runner.backtest_saved(run, ['test']) once for every run, then ['stress_bear']"}, indent=1, default=list))
print("wrote", INIT / "selected_agent_e16.json", "|", REP / "e16_agents_val.md", "|", REP / "e16_val_equity.png")

In [ ]:
# release the GPU once everything above is on Drive (outputs stay visible in this notebook)
if AUTO_DISCONNECT and "google.colab" in sys.modules:
    from google.colab import runtime
    runtime.unassign()